# Dictionaries in Python

A dictionary stores key–value pairs. In this lesson, you will create dictionaries, retrieve and modify values, handle missing keys, and choose suitable ways to process dictionary data.

## Learning objectives
By the end of the lesson, you should be able to:
- Construct dictionaries with literals, `dict()`, `zip()`, and `fromkeys()`.
- Explain the roles of keys, values, hashability, and insertion order.
- Retrieve, add, update, and remove entries.
- Choose between dictionary methods and operators.
- Distinguish a shallow copy from an independent copy of nested objects.
- Iterate over keys, values, and key–value pairs.
- Recognize specialized dictionary-like classes.




## Lesson outline
1. Dictionary fundamentals
2. Construction and valid keys
3. Accessing and building dictionaries
4. Dictionary methods
5. Operators and built-in functions
6. Iteration
7. Specialized dictionary-like classes
8. Review

## 1. Dictionary fundamentals

A dictionary associates each key with a value. Use a key in square brackets to retrieve its value. Dictionaries are mutable: assigning to an existing key replaces its value.

In [2]:
config = {
    "color": "green",
    "width": 42,
    "height": 100,
    "font": "Courier",
}
print(config["color"])
config["font"] = "Helvetica"
print(config)

green
{'color': 'green', 'width': 42, 'height': 100, 'font': 'Helvetica'}


Keys are unique within a dictionary and must be hashable. Values may be any Python objects and may repeat. Dictionaries preserve insertion order in Python 3.7 and later, but ordinary dictionary equality does not depend on that order.

### Dictionaries and object attributes

Dictionaries also help organize Python namespaces and attributes. The following class stores an instance attribute named `value`. Its instance dictionary exposes that attribute. This example does not imply that every Python object has a writable `__dict__`.

In [3]:
class Number:
    def __init__(self, value):
        self.value = value

print(Number(42).__dict__)

{'value': 42}


## 2. Creating dictionaries

A literal places comma-separated `key: value` pairs inside braces. Empty braces create an empty dictionary, not an empty set.

In [4]:
MLB_teams = {
    "Colorado": "Rockies",
    "Chicago": "White Sox",
    "Boston": "Red Sox",
    "Minnesota": "Twins",
    "Milwaukee": "Brewers",
    "Seattle": "Mariners",
}
print(MLB_teams)
print({})
print(dict())

{'Colorado': 'Rockies', 'Chicago': 'White Sox', 'Boston': 'Red Sox', 'Minnesota': 'Twins', 'Milwaukee': 'Brewers', 'Seattle': 'Mariners'}
{}
{}


### Valid keys and hashability

Numbers, strings, and many other objects can be keys. A tuple can be a key only if every item it contains is hashable. Lists are unhashable, including when a list appears inside a tuple. The requirement is hashability, not simply the appearance of an immutable outer container.

In [5]:
print({42: "aaa", 2.78: "bbb", True: "ccc"})
types = {int: 1, float: 2, bool: 3}
print(types)
print(types[float])
print(types[bool])

{42: 'aaa', 2.78: 'bbb', True: 'ccc'}
{<class 'int'>: 1, <class 'float'>: 2, <class 'bool'>: 3}
2
3


In [6]:
try:
    {[1, 2]: "A list as a key?"}
except TypeError as error:
    print(type(error).__name__ + ":", error)

a_dict = {(1, 1): "a", (1, 2): "b", (2, 1): "c", (2, 2): "d"}
print(a_dict[(1, 1)])
print(a_dict[(2, 1)])

try:
    {(1, [1, 1]): "a"}
except TypeError as error:
    print(type(error).__name__ + ":", error)

TypeError: unhashable type: 'list'
a
c
TypeError: unhashable type: 'list'


### Replacing values and repeated keys

Assigning another value to an existing key does not add an entry. If a literal repeats a key, the final value supplied for that key is retained. Predict the number of entries before running each example.

In [7]:
MLB_teams["Chicago"] = "Cubs"
print(MLB_teams)
print("Entries:", len(MLB_teams))

repeated_keys = {"Chicago": "White Sox", "Chicago": "Cubs"}
print(repeated_keys)

{'Colorado': 'Rockies', 'Chicago': 'Cubs', 'Boston': 'Red Sox', 'Minnesota': 'Twins', 'Milwaukee': 'Brewers', 'Seattle': 'Mariners'}
Entries: 6
{'Chicago': 'Cubs'}


### Values can have different types

Unlike keys, values do not have to be hashable. A dictionary may contain lists, sets, numbers, and custom objects as values. Set display order and default object representations should not be treated as fixed expected-output strings.

In [8]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

mixed_values = {
    "colors": ["red", "green", "blue"],
    "plugins": {"py_code", "dev_sugar", "fasting_py"},
    "timeout": 3,
    "position": Point(42, 21),
}
print(mixed_values["colors"])
print(sorted(mixed_values["plugins"]))
print(mixed_values["timeout"])
print((mixed_values["position"].x, mixed_values["position"].y))
print({0: "a", 1: "a", 2: "a", 3: "a"})

['red', 'green', 'blue']
['dev_sugar', 'fasting_py', 'py_code']
3
(42, 21)
{0: 'a', 1: 'a', 2: 'a', 3: 'a'}


### The `dict()` constructor

Construct a dictionary using keyword arguments, another mapping, or an iterable of two-item key–value pairs. Keyword argument names must be valid identifiers. The examples below restore Chicago’s value to `White Sox`.

In [9]:
MLB_teams = dict(
    Colorado="Rockies", Chicago="White Sox", Boston="Red Sox",
    Minnesota="Twins", Milwaukee="Brewers", Seattle="Mariners",
)
print(MLB_teams)

{'Colorado': 'Rockies', 'Chicago': 'White Sox', 'Boston': 'Red Sox', 'Minnesota': 'Twins', 'Milwaukee': 'Brewers', 'Seattle': 'Mariners'}


In [10]:
MLB_teams = dict([
    ("Colorado", "Rockies"), ("Chicago", "White Sox"),
    ("Boston", "Red Sox"), ("Minnesota", "Twins"),
    ("Milwaukee", "Brewers"), ("Seattle", "Mariners"),
])
print(MLB_teams)

{'Colorado': 'Rockies', 'Chicago': 'White Sox', 'Boston': 'Red Sox', 'Minnesota': 'Twins', 'Milwaukee': 'Brewers', 'Seattle': 'Mariners'}


### Combining `zip()` and `dict()`

`zip()` pairs corresponding items from two iterables. The order of the input sequences determines which key is paired with which value.

In [11]:
places = ["Colorado", "Chicago", "Boston", "Minnesota", "Milwaukee", "Seattle"]
teams = ["Rockies", "White Sox", "Red Sox", "Twins", "Brewers", "Mariners"]
print(dict(zip(places, teams)))

{'Colorado': 'Rockies', 'Chicago': 'White Sox', 'Boston': 'Red Sox', 'Minnesota': 'Twins', 'Milwaukee': 'Brewers', 'Seattle': 'Mariners'}


### Initializing entries with `fromkeys()`

Use `dict.fromkeys(keys, default)` when several keys need the same initial value. Here, each stock count starts at zero.

In [12]:
inventory = dict.fromkeys(["apple", "orange", "banana", "mango"], 0)
print(inventory)

{'apple': 0, 'orange': 0, 'banana': 0, 'mango': 0}


### Check your understanding

1. Why is a tuple containing a list unsuitable as a dictionary key?
2. Why does replacing Chicago’s team not increase the dictionary’s length?
3. What determines the correspondence between keys and values in the `zip()` example?

## 3. Accessing dictionary values

Square-bracket lookup uses a key, not an ordinal position. Looking up a missing key raises `KeyError`.

In [13]:
print(MLB_teams["Minnesota"])
print(MLB_teams["Colorado"])
try:
    MLB_teams["Indianapolis"]
except KeyError as error:
    print(type(error).__name__ + ":", error)

Twins
Rockies
KeyError: 'Indianapolis'


### Accessing nested data

Apply one lookup or indexing operation at each level. The `children` value is a list, while the `pets` value is another dictionary.

In [14]:
person = {
    "first_name": "John", "last_name": "Doe", "age": 35,
    "spouse": "Jane", "children": ["Ralph", "Betty", "Bob"],
    "pets": {"dog": "Frieda", "cat": "Sox"},
}
print(person["children"][0])
print(person["children"][2])
print(person["pets"]["dog"])
print(person["pets"]["cat"])

Ralph
Bob
Frieda
Sox


### Building dictionaries incrementally

Start with an empty dictionary and assign entries as data becomes available. A loop or comprehension can generate entries systematically.

In [15]:
person = {}
person["first_name"] = "John"
person["last_name"] = "Doe"
person["age"] = 35
person["spouse"] = "Jane"
person["children"] = ["Ralph", "Betty", "Bob"]
person["pets"] = {"dog": "Frieda", "cat": "Sox"}
print(person)

{'first_name': 'John', 'last_name': 'Doe', 'age': 35, 'spouse': 'Jane', 'children': ['Ralph', 'Betty', 'Bob'], 'pets': {'dog': 'Frieda', 'cat': 'Sox'}}


In [16]:
squares = {}
for integer in range(1, 10):
    squares[integer] = integer**2
print(squares)

squares = {integer: integer**2 for integer in range(1, 10)}
print(squares)

{1: 1, 2: 4, 3: 9, 4: 16, 5: 25, 6: 36, 7: 49, 8: 64, 9: 81}
{1: 1, 2: 4, 3: 9, 4: 16, 5: 25, 6: 36, 7: 49, 8: 64, 9: 81}


## 4. Dictionary methods

### Retrieving optional values with `.get()`

`.get(key)` returns `None` when the key is absent. Provide a second argument to choose another default. Unlike bracket lookup, a missing key does not raise `KeyError`; `.get()` also does not insert an entry.

In [17]:
inventory = {"apple": 100, "orange": 80, "banana": 100}
print(inventory.get("apple"))
print(inventory.get("mango"))
print(inventory.get("mango", 0))
print(inventory)

100
None
0
{'apple': 100, 'orange': 80, 'banana': 100}


### Dictionary views

`.keys()`, `.values()`, and `.items()` return views of keys, values, and key–value pairs. These views are dynamic: changes to the dictionary are visible through an existing view. Values can repeat.

In [18]:
print(inventory.keys())
print(inventory.values())
print(inventory.items())
keys_view = inventory.keys()
inventory["mango"] = 0
print(keys_view)

dict_keys(['apple', 'orange', 'banana'])
dict_values([100, 80, 100])
dict_items([('apple', 100), ('orange', 80), ('banana', 100)])
dict_keys(['apple', 'orange', 'banana', 'mango'])


### Inserting defaults with `.setdefault()`

For an existing key, return the current value without replacing it. For a missing key, insert and return the default. If no default is supplied, use `None`. Compare this behavior with `.get()`.

In [19]:
inventory = {"apple": 100, "orange": 80}
print(inventory.setdefault("apple"))
print(inventory.setdefault("mango"))
print(inventory)
print(inventory.setdefault("banana", 0))
print(inventory)

100
None
{'apple': 100, 'orange': 80, 'mango': None}
0
{'apple': 100, 'orange': 80, 'mango': None, 'banana': 0}


### Updating multiple entries

`.update()` modifies the existing dictionary. Existing keys receive new values and missing keys are added. Inputs can include another dictionary, an iterable of pairs, or keyword arguments.

In [20]:
config = {"color": "green", "width": 42, "height": 100, "font": "Courier"}
user_config = {
    "path": "/home", "color": "red", "font": "Arial", "position": (200, 100),
}
config.update(user_config)
print(config)
config.update([("width", 200), ("api_key", 1234)])
print(config)
config.update(color="yellow", script="__main__.py")
print(config)

{'color': 'red', 'width': 42, 'height': 100, 'font': 'Arial', 'path': '/home', 'position': (200, 100)}
{'color': 'red', 'width': 200, 'height': 100, 'font': 'Arial', 'path': '/home', 'position': (200, 100), 'api_key': 1234}
{'color': 'yellow', 'width': 200, 'height': 100, 'font': 'Arial', 'path': '/home', 'position': (200, 100), 'api_key': 1234, 'script': '__main__.py'}


### Removing entries with `.pop()` and `del`

`.pop(key)` removes the entry and returns its value. A missing key raises `KeyError` unless a default is provided. `del dictionary[key]` removes the entry without returning its value.

In [21]:
inventory = {"apple": 100, "orange": 80, "banana": 100}
print(inventory.pop("apple"))
print(inventory)
try:
    inventory.pop("mango")
except KeyError as error:
    print(type(error).__name__ + ":", error)
print(inventory.pop("mango", 0))
del inventory["banana"]
print(inventory)

100
{'orange': 80, 'banana': 100}
KeyError: 'mango'
0
{'orange': 80}


### Removing the last inserted pair

`.popitem()` returns and removes the last inserted remaining key–value pair. This last-in, first-out behavior is guaranteed in Python 3.7 and later. Calling it on an empty dictionary raises `KeyError`.

In [22]:
inventory = {"apple": 100, "orange": 80, "banana": 100}
print(inventory.popitem())
print(inventory)
print(inventory.popitem())
print(inventory)
print(inventory.popitem())
print(inventory)
try:
    inventory.popitem()
except KeyError as error:
    print(type(error).__name__ + ":", error)

('banana', 100)
{'apple': 100, 'orange': 80}
('orange', 80)
{'apple': 100}
('apple', 100)
{}
KeyError: 'popitem(): dictionary is empty'


### Clearing a dictionary

`.clear()` removes every entry from the dictionary.

In [23]:
inventory = {"apple": 100, "orange": 80, "banana": 100}
inventory.clear()
print(inventory)

{}


### Understanding shallow copies

`.copy()` creates a different outer dictionary with the same key–value pairs. It does not recursively copy nested objects. Use `is` to test identity and `==` to test equality.

In [24]:
inventory = {"apple": 100, "orange": 80, "banana": 100}
inventory_copy = inventory.copy()
print(inventory_copy)
print("Same object:", inventory_copy is inventory)
print("Equal content:", inventory_copy == inventory)

{'apple': 100, 'orange': 80, 'banana': 100}
Same object: False
Equal content: True


Both dictionaries below reference the same nested children list. Changing that list through either dictionary is visible through the other. The final assignment restores its initial content.

In [26]:
print(person)
person_copy = person.copy()
print("Same outer dictionary:", person_copy is person)
print("Same children list:", person_copy["children"] is person["children"])
person_copy["children"][0] = "Ralph (updated)"
print(person["children"])
person_copy["children"][0] = "Ralph"
print(person["children"])
print(person_copy["children"])

{'first_name': 'John', 'last_name': 'Doe', 'age': 35, 'spouse': 'Jane', 'children': ['Ralph', 'Betty', 'Bob'], 'pets': {'dog': 'Frieda', 'cat': 'Sox'}}
Same outer dictionary: False
Same children list: True
['Ralph (updated)', 'Betty', 'Bob']
['Ralph', 'Betty', 'Bob']
['Ralph', 'Betty', 'Bob']


## 5. Dictionary operators

### Membership

`key in dictionary` tests keys. To test values, use `.values()`. To test a complete `(key, value)` pair, use `.items()`. Calling `.keys()` is not necessary for a key-membership test.

In [ ]:
print("Milwaukee" in MLB_teams)
print("Indianapolis" in MLB_teams)
print("Indianapolis" not in MLB_teams)
print("Milwaukee" in MLB_teams.keys())
print("Rockies" in MLB_teams.values())
print("Rockies" not in MLB_teams.values())
print(("Boston", "Red Sox") in MLB_teams.items())
print(("Boston", "Red Sox") not in MLB_teams.items())

True
False
True
True
True
False
True
False


### Equality

Ordinary dictionaries compare their key–value content without requiring the same insertion order. Lists, in contrast, require corresponding elements to appear in the same order for equality.

In [ ]:
print([1, 2, 3] == [3, 2, 1])
print({1: 1, 2: 2, 3: 3} == {3: 3, 2: 2, 1: 1})
print([1, 2, 3] != [3, 2, 1])
print({1: 1, 2: 2, 3: 3} != {3: 3, 2: 2, 1: 1})

False
True
True
False


### Merging dictionaries

`left | right` creates a merged dictionary. If a key appears on both sides, its value comes from the right. `left |= right` instead updates the existing dictionary.

In [ ]:
default_config = {
    "color": "green", "width": 42, "height": 100, "font": "Courier",
}
user_config = {
    "path": "/home", "color": "red", "font": "Arial", "position": (200, 100),
}
config = default_config | user_config
print(config)
print("Defaults unchanged:", default_config)

{'color': 'red', 'width': 42, 'height': 100, 'font': 'Arial', 'path': '/home', 'position': (200, 100)}
Defaults unchanged: {'color': 'green', 'width': 42, 'height': 100, 'font': 'Courier'}


In [ ]:
config = {"color": "green", "width": 42, "height": 100, "font": "Courier"}
config |= user_config
print(config)

{'color': 'red', 'width': 42, 'height': 100, 'font': 'Arial', 'path': '/home', 'position': (200, 100)}


### Built-in functions

`len()` counts entries. When a dictionary itself is passed to `all()`, `any()`, or `max()`, these functions process its keys. Use `.values()` to process the values instead. Here, fruit names are nonempty strings and the initial counts are positive integers.

In [ ]:
inventory = {"apple": 100, "orange": 80, "banana": 100}
print("Number of entries:", len(inventory))
print("All keys truthy:", all(inventory))
print("Any key truthy:", any(inventory))
print("Maximum key:", max(inventory))
print("All values truthy:", all(inventory.values()))
print("Any value truthy:", any(inventory.values()))
print("Maximum value:", max(inventory.values()))

Number of entries: 3
All keys truthy: True
Any key truthy: True
Maximum key: orange
All values truthy: True
Any value truthy: True
Maximum value: 100


Now initialize every count to zero. Predict which truth tests will change. Remember that numeric zero is false in a truth test, while a nonempty string is true.

In [ ]:
inventory = dict.fromkeys(["apple", "orange", "banana", "mango"], 0)
print("All keys truthy:", all(inventory))
print("Any key truthy:", any(inventory))
print("All values truthy:", all(inventory.values()))
print("Any value truthy:", any(inventory.values()))

All keys truthy: True
Any key truthy: True
All values truthy: False
Any value truthy: False


## 6. Iterating over dictionaries

Iterating directly over a dictionary yields its keys. Explicitly iterating over `.keys()` yields the same keys.

In [ ]:
for place in MLB_teams:
    print(place)

Colorado
Chicago
Boston
Minnesota
Milwaukee
Seattle


In [ ]:
for place in MLB_teams.keys():
    print(place)

Colorado
Chicago
Boston
Minnesota
Milwaukee
Seattle


Use `.values()` when only values are needed. Use `.items()` and unpack each pair when both keys and values are needed.

In [ ]:
for team in MLB_teams.values():
    print(team)

Rockies
White Sox
Red Sox
Twins
Brewers
Mariners


In [ ]:
for place, team in MLB_teams.items():
    print(place, "->", team)

Colorado -> Rockies
Chicago -> White Sox
Boston -> Red Sox
Minnesota -> Twins
Milwaukee -> Brewers
Seattle -> Mariners


## 7. Specialized dictionary-like classes

The standard-library `collections` module includes classes for more specific needs.

| Class | Main role |
|---|---|
| `OrderedDict` | An order-oriented dictionary subclass with capabilities beyond ordinary dictionaries. |
| `Counter` | A dictionary subclass designed for counting. |
| `defaultdict` | A dictionary subclass that can supply values for missing keys through a default factory. |

Ordinary dictionaries already preserve insertion order in Python 3.7 and later. Insertion order alone therefore does not distinguish an ordinary dictionary from an `OrderedDict`.

### Creating custom dictionary-like classes
Two inheritance-based approaches are subclassing `dict` and subclassing `collections.UserDict`. Directly overriding `dict` behavior requires care because inherited operations do not necessarily use overridden methods in the way you might expect. `UserDict` provides a wrapper around a regular dictionary and is often a more convenient starting point for customization.

This section introduces the available choices; implementing a custom mapping is beyond this lesson.

## 8. Review

1. What distinguishes a key from a value? Which must be hashable?
2. Why does replacing a value not necessarily increase the dictionary’s length?
3. How do square-bracket lookup, `.get()`, and `.setdefault()` handle a missing key?
4. How do you test membership among keys, values, and complete pairs?
5. Can dictionaries with different insertion orders be equal?
6. How do `.update()`, `|`, and `|=` differ?
7. What can remain shared after a shallow copy?
8. Which iteration form provides both keys and values?
9. Which specialized class supports counting? Which supports missing-key defaults?

### Answer guide
1. A key identifies an associated value. Keys must be hashable; values need not be.
2. Assigning to an existing key replaces its previous value.
3. Bracket lookup raises `KeyError`; `.get()` returns a default without insertion; `.setdefault()` inserts and returns the default.
4. Use the dictionary itself, `.values()`, and `.items()`, respectively.
5. Yes. Ordinary dictionary equality compares key–value content.
6. `.update()` and `|=` modify the existing dictionary; `|` creates a merged result.
7. Nested objects such as lists may still be shared.
8. Iterate over `.items()` and unpack each pair.
9. `Counter` supports counting; `defaultdict` supports missing-key defaults.

